In [1]:
import sys
sys.path.append('../lib')
import numpy as np
from scipy.ndimage import gaussian_filter
import pymaster as nmt
import matplotlib.pyplot as plt

In [2]:
# Inputs

nside = 64
lmax = 2*nside-1
fsky = 0.7
scale = 10
Nlbin = 1
dusttype = 'b'
synctype = 'b'
Pathload = '/pscratch/sd/s/svinzl/B_modes_project/'
N = 250
cov_type = 'Nmt-fg'
kw = ''
kws = ''
dusttype_cov = dusttype
synctype_cov = synctype
gaussbeam = True
bandpass = True
Ngrid = 50
cmb_e2e = True
n_iter = 3
adaptative = False
pl_moms = False
nu0d = 402
nu0s = 40
sigma = 10 # sigma for smoothing moments

if gaussbeam:
    kws += '_gaussbeam'
if bandpass:
    kws += '_bandpass'

if adaptative:
    kw += '_adaptative'

# Instrument

instr_name = 'litebird_full'
instr = np.load('../lib/instr_dict/%s.npy'%instr_name, allow_pickle=True).item()
freq = instr['frequencies']
N_freqs = len(freq)
Ncross = int(N_freqs * (N_freqs+1)/2)

if bandpass:
    bw = instr['bandwidths']
    freq_grids = np.zeros((N_freqs, Ngrid))
    for i in range(N_freqs):
        freq_grids[i] = np.geomspace(freq[i]-bw[i]/2, freq[i]+bw[i]/2, Ngrid)
    freq = freq_grids

freq_pairs = np.array([(i, j) for i in range(N_freqs) for j in range(i, N_freqs)])
nu_i = freq[freq_pairs[:, 0]]
nu_j = freq[freq_pairs[:, 1]]

# Binning scheme

b = nmt.NmtBin.from_lmax_linear(lmax, nlb=Nlbin, is_Dell=True)
leff = b.get_effective_ells()
Nbins = len(leff)

# Data and precomputed results

data = np.load(Pathload+'/power_spectra/DLcross_nside%s_fsky%s_scale%s_Nlbin%s_d%ss%sc' % (nside, fsky, scale, Nlbin, dusttype, synctype)+kws+'.npy')[:, :, :Nbins]
#covmat = np.load(Pathload+'covariances/cov_%s_nside%s_fsky%s_scale%s_Nlbin%s_d%ss%sc' % (cov_type, nside, fsky, scale, Nlbin, dusttype_cov, synctype_cov)+kws+'.npy')[:Ncross*Nbins, :Ncross*Nbins]
results = np.load('../best_fits/results_d%ss%s_%s_scale%s_Nlbin%s_%s%s_ds_o1bts%s_analytical.npy' % (dusttype, synctype, fsky, scale, Nlbin, cov_type, kws, kw), allow_pickle=True).item()

In [3]:
# Smooth fitted moments

keys = list(results)
Nmoms = len(keys) - 5

results_smoothed = {}
for par in keys:
    DLmom = results[par]
    if par not in ['A', 'As', 'Asd', 'cmb', 'beta_d', 'T_d', 'beta_s', 'chi2r']:
        for k in range(N):
            DLmom[:, k] = gaussian_filter(DLmom[:, k], sigma, mode='constant')

    results_smoothed[par] = DLmom

s_fg = np.zeros((N, Nmoms*Nbins))
for k in range(N):
    idx = 0
    for par in keys:
        if par not in ['cmb', 'beta_d', 'T_d', 'beta_s', 'chi2r']:
            s_fg[k, np.arange(Nbins)*Nmoms + idx] = results_smoothed[par][:, k]
            idx += 1

In [5]:
# Compute weight matrices

betabar = np.mean(results['beta_d'])
tempbar = np.mean(results['T_d'])
betasbar = np.mean(results['beta_s'])

comp_fg = [['A', 'As', 'Asd', 'Aw1b', 'Aw1t', 'w1bw1b', 'w1tw1t', 'w1bw1t', 'Asw1bs', 'w1bsw1bs', 'Asw1b', 'Asw1t', 'Aw1bs', 'w1bw1bs', 'w1tw1bs']
       for i in range(Nbins)]
W_fg = an.gauss_like(freq, leff, covmat, comp_fg, betabar, tempbar, betasbar, nu0d, nu0s).W

data_cmb = np.zeros_like(data)
for k in range(N):
    data_cmb[k] = data[k]

comp = [['cmb']
       for i in range(Nbins)]
gauss = an.gauss_like(freq, leff, covmat, comp, betabar, tempbar, betasbar, nu0d, nu0s)

results_final = gauss.

In [8]:
data.shape

(250, 253, 126)